# BigQuery ML Financial Forecasting Analysis

## Project Overview

This notebook implements a comprehensive BigQuery ML (BQML) time series forecasting solution for financial data analysis. It leverages ARIMA_PLUS models to generate accurate forecasts across different organizational hierarchies including groups, divisions, regions, and accounts.

### Key Features

- **Automated Model Creation**: Reusable functions for creating forecast models across any attribute
- **Time Series Decomposition**: Trend, seasonal, and holiday effect analysis
- **Comprehensive Diagnostics**: Model evaluation metrics and performance analysis
- **Interactive Visualizations**: Historical vs forecast comparisons with confidence intervals
- **Multi-level Analysis**: Support for group, division, region, and account-level forecasting

### Business Context

This analysis helps financial teams:
- Forecast future financial amounts by organizational structure
- Identify trends and seasonal patterns in financial data
- Make data-driven budgeting and planning decisions
- Understand variance drivers through decomposition analysis

---

## Environment Setup and Installation

### Required Packages

Install the necessary Python packages for BigQuery integration and data visualization:

In [ ]:
!pip install google-cloud-bigquery matplotlib seaborn numpy pandas

### Import Libraries and Authentication

Import all required libraries and authenticate with Google Cloud:

In [ ]:
import bigframes.pandas as bpd
import pandas as pd
from google.cloud import bigquery
import google.auth
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np

# Get project ID automatically
_, PROJECT_ID = google.auth.default()
print(f"Project ID: {PROJECT_ID}")

### Verify Authentication

Confirm that Google Cloud authentication is working correctly:

In [ ]:
# Check if authentication is working
try:
    client = bigquery.Client(project=PROJECT_ID)
    print("Authentication successful!")
except Exception as e:
    print(f"Authentication failed: {e}")

## Dataset cleanup (when major changes happen in naming)

In [ ]:
def delete_dataset_objects(
    project_id: str,
    dataset_id: str,
    delete_tables: bool = True,
    delete_views: bool = False,
    delete_models: bool = False,
):
    statements = []

    if delete_tables or delete_views:
        object_filter = []
        if delete_tables:
            object_filter.append("'BASE TABLE'")
        if delete_views:
            object_filter.append("'VIEW'")
            object_filter.append("'MATERIALIZED VIEW'")

        tables_sql = f"""
        SELECT
          CASE
            WHEN table_type = 'BASE TABLE' THEN FORMAT(
              'DROP TABLE `%s.%s.%s`',
              table_catalog,
              table_schema,
              table_name
            )
            WHEN table_type = 'VIEW' THEN FORMAT(
              'DROP VIEW `%s.%s.%s`',
              table_catalog,
              table_schema,
              table_name
            )
            WHEN table_type = 'MATERIALIZED VIEW' THEN FORMAT(
              'DROP MATERIALIZED VIEW `%s.%s.%s`',
              table_catalog,
              table_schema,
              table_name
            )
          END AS stmt
        FROM `{project_id}.{dataset_id}.INFORMATION_SCHEMA.TABLES`
        WHERE table_type IN ({", ".join(object_filter)})
        """
        statements.extend(row.stmt for row in client.query(tables_sql).result() if row.stmt)

    if delete_models:
        models_sql = f"""
        SELECT FORMAT(
          'DROP MODEL `%s.%s.%s`',
          model_catalog,
          model_schema,
          model_name
        ) AS stmt
        FROM `{project_id}.{dataset_id}.MODELS`
        """
        statements.extend(row.stmt for row in client.query(models_sql).result() if row.stmt)

    for stmt in statements:
        client.query(stmt).result()
        print(f"Executed: {stmt}")

    print(f"Completed cleanup for {project_id}.{dataset_id}")

In [ ]:
delete_dataset_objects(
    PROJECT_ID,
    "forecasting_us",
    delete_tables=True,
    delete_views=True,
    delete_models=False,
)

## Data Loading and Exploration

### Load Source Data

Load the general ledger summary data from BigQuery:

In [ ]:
df_summ = bpd.read_gbq(f'{PROJECT_ID}.ai_financial_dlp.XXC_GL_SUMMARY')
df_summ.columns

## Data Preparation and Transformation

### Create Forecasting Dataset

Transform the raw financial data into a clean dataset suitable for time series forecasting:

**Key Transformations:**
- Filter for positive amounts only (credits/revenue)
- Join general ledger data with facility information
- Include only valid ledger entries (ledger_id = '1')
- Ensure data quality with non-null effective dates

**Resulting Dataset Features:**
- Financial amounts and dates
- Organizational hierarchy (group, division, region)
- Facility information and vendor details
- Account and sub-account classifications

In [ ]:
result = client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.forecasting_us.POSITIVE_AMOUNTS` AS
SELECT
  summary.location,
  summary.ledger_id,
  summary.amount,
  summary.period_name,
  -- Convert period_name (e.g., "MAY-25", "APR-25") to first day of month in yyyy-mm-dd format
  CASE
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'JAN' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-01-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'FEB' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-02-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'MAR' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-03-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'APR' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-04-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'MAY' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-05-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'JUN' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-06-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'JUL' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-07-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'AUG' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-08-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'SEP' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-09-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'OCT' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-10-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'NOV' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-11-01'))
    WHEN UPPER(SUBSTR(summary.period_name, 1, 3)) = 'DEC' THEN DATE(CONCAT('20', SUBSTR(summary.period_name, 5, 2), '-12-01'))
    ELSE NULL
  END AS PERIOD_NAME_DS,
  summary.JOURNAL_POSTED_DATE,
  summary.line_description,
  summary.category,
  summary.super_category,
  summary.commodity,
  summary.vendor_name,
  summary.vendor_id,
  summary.account,
  summary.account_name,
  summary.sub_account,
  summary.sub_account_name,
  facility.facility_id,
  facility.facility_description,
  facility.facility_common_name,
  facility.palmer_vp,
  facility.palmer_vp_desc,
  facility.group_vp,
  facility.group_vp_desc,
  facility.division,
  facility.division_desc,
  facility.region,
  facility.region_desc,
  facility.facility_type,
  facility.legal_entity,
  facility.legal_entity_desc,
  facility.facility_city,
  facility.facility_state
FROM
  `{PROJECT_ID}.ai_financial_dlp.XXC_GL_SUMMARY` AS summary
JOIN
  `{PROJECT_ID}.ai_financial_dlp.XXC_GL_DIV_REG_FAC` AS facility
  ON summary.location = facility.facility_id
WHERE
  summary.amount >= 0
  AND summary.period_name IS NOT NULL
  AND TRIM(summary.ledger_id)='1'
ORDER BY location;""").result()

In [ ]:
pos_ammount = bpd.read_gbq(f'{PROJECT_ID}.forecasting_us.POSITIVE_AMOUNTS', use_cache=False)

In [ ]:
pos_ammount["period_name_ds"].unique()

## Core Forecasting Functions

### Main Model Creation Function

This is the core function that creates BigQuery ML ARIMA_PLUS forecasting models for any organizational attribute.

**Function Features:**
- **Automated Data Processing**: Filters, aggregates, and prepares time series data
- **Model Creation**: Builds ARIMA_PLUS models with automatic parameter selection
- **Forecast Generation**: Produces future predictions with confidence intervals
- **Performance Evaluation**: Calculates model diagnostics and quality metrics
- **Visualization Preparation**: Creates combined datasets for plotting
- **Time Series Decomposition**: Breaks down into trend, seasonal, and holiday effects

**Parameters:**
- `project_id`: Google Cloud project ID
- `attribute_col`: Column to group by (e.g., 'group_vp', 'division')
- `attribute_values`: List of values to forecast
- `forecast_horizon`: Number of days to forecast (default: 7)

**Returns:**
- Dictionary containing model information and evaluation results

In [ ]:
def create_forecast_model(project_id, attribute_col, attribute_values, forecast_horizon=1):
    """
    Reusable function to create BigQuery ML forecast models for any attribute.

    Parameters:
        attribute_col: Column name to group by (e.g., 'group_vp', 'palmer_vp', 'division', 'region')
        attribute_values: List of values to forecast (e.g., ['GV012', 'GV009'])
        forecast_horizon: Number of days to forecast (default 7)
    """

    # Build model name
    clean_col = attribute_col.replace('.', '_').replace('-', '_').upper()
    model_name = f"GROUP_FORECAST_MODEL_{clean_col}"

    # Format attribute values for SQL IN clause
    values_str = ', '.join([f"'{v}'" for v in attribute_values])

    dataset = f'{project_id}.forecasting_us'
    source_table = f'{dataset}.POSITIVE_AMOUNTS'

    # Step 1: Create forecast data
    print(f"Step 1: Creating forecast data for {attribute_col.upper()} = [{values_str}]...")
    step1_sql = f"""
    CREATE OR REPLACE TABLE `{dataset}.{model_name}_FORECAST_DATA` AS
    WITH filtered_data AS (
      SELECT
        period_name_ds as DS,
        amount as Y,
        {attribute_col.upper()}
      FROM `{source_table}`
      WHERE {attribute_col} IN ({values_str})
        AND period_name_ds IS NOT NULL
        AND amount IS NOT NULL
        AND amount >= 0
    ),
    aggregated_data AS (
      SELECT
        DS,
        {attribute_col.upper()},
        SUM(y) as Y
      FROM filtered_data
      GROUP BY DS, {attribute_col.upper()}
    )
    SELECT
      DS,
      Y,
      {attribute_col}
    FROM aggregated_data
    ORDER BY {attribute_col.upper()}, DS
    """
    client.query(step1_sql).result()
    print("  Done.")

    # Step 2: Create ARIMA_PLUS model
    print(f"Step 2: Creating ARIMA_PLUS model...")
    step2_sql = f"""
    CREATE OR REPLACE MODEL `{dataset}.{model_name}`
    OPTIONS(
      model_type = 'ARIMA_PLUS',
      time_series_timestamp_col = 'DS',
      time_series_data_col = 'Y',
      time_series_id_col = '{attribute_col.upper()}',
      data_frequency = 'DAILY',
      auto_arima = TRUE,
      holiday_region = 'US'
    ) AS
    SELECT DS, Y, {attribute_col.upper()}
    FROM `{dataset}.{model_name}_FORECAST_DATA`
    """
    client.query(step2_sql).result()
    print("  Done.")

    # Step 3: Generate forecasts
    print(f"Step 3: Generating {forecast_horizon}-month(s) forecasts...")
    step3_sql = f"""
    CREATE OR REPLACE TABLE `{dataset}.{model_name}_FORECAST_RESULTS` AS
    SELECT
      FORECAST_TIMESTAMP,
      FORECAST_VALUE,
      CONFIDENCE_LEVEL,
      PREDICTION_INTERVAL_LOWER_BOUND,
      PREDICTION_INTERVAL_UPPER_BOUND,
      {attribute_col.upper()}
    FROM ML.FORECAST(
      MODEL `{dataset}.{model_name}`,
      STRUCT({forecast_horizon} AS HORIZON)
    )
    """
    client.query(step3_sql).result()
    print("  Done.")

    # Step 4: Model evaluation
    print(f"Step 4: Evaluating model...")

    step4_sql = f"""
    CREATE OR REPLACE TABLE `{dataset}.{model_name}_EVALUATION` AS
    SELECT *
    FROM ML.ARIMA_EVALUATE(MODEL `{dataset}.{model_name}`)
    """
    client.query(step4_sql).result()

    evaluation = client.query(f"SELECT * FROM `{dataset}.{model_name}_EVALUATION`").to_dataframe()
    print("  Done.")

    # Step 5: Create visualization data
    print(f"Step 5: Creating visualization data...")
    step5_sql = f"""
    CREATE OR REPLACE TABLE `{dataset}.{model_name}_VISUALIZATION_DATA` AS
    WITH COMBINED_DATA AS (
      SELECT
        DS as DATE,
        Y as VALUE,
        {attribute_col},
        'Historical' as DATA_TYPE,
        NULL as LOWER_BOUND,
        NULL as UPPER_BOUND
      FROM `{dataset}.{model_name}_FORECAST_DATA`

      UNION ALL

      SELECT
        CAST(forecast_timestamp AS DATE) as DATE,
        FORECAST_VALUE as VALUE,
        {attribute_col},
        'Forecast' as DATA_TYPE,
        PREDICTION_INTERVAL_LOWER_BOUND as LOWER_BOUND,
        PREDICTION_INTERVAL_UPPER_BOUND as UPPER_BOUND
      FROM `{dataset}.{model_name}_FORECAST_RESULTS`
    )
    SELECT
      DATE,
      VALUE,
      {attribute_col.upper()},
      DATA_TYPE,
      LOWER_BOUND,
      UPPER_BOUND
    FROM COMBINED_DATA
    ORDER BY {attribute_col.upper()}, DATE
    """
    client.query(step5_sql).result()
    print("  Done.")

    # Step 6: Create decomposition
    print(f"Step 6: Creating time series decomposition...")
    step6_sql = f"""
    CREATE OR REPLACE TABLE `{dataset}.{model_name}_DECOMPOSITION` AS
    SELECT
      {attribute_col},
      TIME_SERIES_TIMESTAMP,
      TIME_SERIES_DATA,
      TREND,
      SEASONAL_PERIOD_YEARLY,
      SEASONAL_PERIOD_MONTHLY,
      SEASONAL_PERIOD_WEEKLY,
      seasonal_period_daily,
      HOLIDAY_EFFECT,
      RESIDUAL
    FROM ML.EXPLAIN_FORECAST(
      MODEL `{dataset}.{model_name}`,
      STRUCT({forecast_horizon} AS HORIZON)
    )
    """
    client.query(step6_sql).result()
    print("  Done.")

    # Summary
    print(f"\n{'='*60}")
    print(f"Forecast model created successfully!")
    print(f"  Attribute: {attribute_col.upper()}")
    print(f"  Values: {values_str}")
    print(f"  Horizon: {forecast_horizon} month(s)")
    print(f"  Tables created:")
    print(f"    - {dataset}.{model_name}_FORECAST_DATA")
    print(f"    - {dataset}.{model_name}_FORECAST_RESULTS")
    print(f"    - {dataset}.{model_name}_EVALUATION")
    print(f"    - {dataset}.{model_name}_VISUALIZATION_DATA")
    print(f"    - {dataset}.{model_name}_DECOMPOSITION")
    print(f"  Model: {dataset}.{model_name}")
    print(f"{'='*60}")

    return {
        'model_name': model_name,
        'dataset': dataset,
        'attribute_col': attribute_col.upper(),
        'attribute_values': attribute_values,
        'forecast_horizon': forecast_horizon,
        'evaluation': evaluation
    }

### Visualization and Analysis Functions

These functions provide comprehensive analysis and visualization capabilities for the forecasting models.

#### Forecast Comparison Plot
- **Historical Data**: Solid lines showing actual past values
- **Forecast Values**: Dashed lines showing predicted future values
- **Confidence Intervals**: Shaded areas showing prediction uncertainty
- **Multi-series Support**: Different colors for each attribute value

#### Model Diagnostics Analysis
- **AIC (Akaike Information Criterion)**: Model quality assessment (lower is better)
- **Log Likelihood**: Model fit measure (higher is better)
- **Variance**: Model variability analysis

#### Time Series Decomposition
- **Trend Component**: Long-term direction and patterns
- **Seasonal Components**: Yearly, weekly, and daily patterns
- **Holiday Effects**: Impact of holidays on the time series
- **Residuals**: Unexplained variation after accounting for other components

#### Summary Statistics
- **Historical vs Forecast Comparison**: Average amounts and percent changes
- **Data Quality Metrics**: Number of data points and standard deviations
- **Performance Indicators**: Model accuracy and reliability measures

In [ ]:
def plot_forecast_comparison(model_info):
    """Plot forecast comparison using model info from create_forecast_model"""
    attr_col = model_info['attribute_col']
    model_name = model_info['model_name']
    dataset = model_info['dataset']
    viz_query = f"""
    SELECT date, value, {attr_col}, data_type, lower_bound, upper_bound
    FROM `{dataset}.{model_name}_visualization_data`
    ORDER BY {attr_col}, date
    """
    viz_data = client.query(viz_query).to_dataframe()
    viz_data['date'] = pd.to_datetime(viz_data['date'])

    plt.figure(figsize=(15, 8))
    groups = viz_data[attr_col].unique()
    colors = plt.cm.tab10.colors

    for i, group in enumerate(groups):
        group_data = viz_data[viz_data[attr_col] == group]
        color = colors[i % len(colors)]

        historical = group_data[group_data['data_type'] == 'Historical']
        if not historical.empty:
            plt.plot(historical['date'], historical['value'],
                    label=f'{group} - Historical', color=color, alpha=0.7, linewidth=2)

        forecast = group_data[group_data['data_type'] == 'Forecast']
        if not forecast.empty:
            plt.plot(forecast['date'], forecast['value'],
                    label=f'{group} - Forecast', color=color, linestyle='--', linewidth=2)
            plt.fill_between(forecast['date'], forecast['lower_bound'], forecast['upper_bound'],
                           alpha=0.2, color=color, label=f'{group} - CI')

    ax = plt.gca()
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    plt.title(f'Forecast Comparison: {attr_col}', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Amount ($)', fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    return viz_data


def analyze_model_diagnostics(model_info):
    """Analyze model diagnostics"""
    model_name = model_info['model_name']
    dataset = model_info['dataset']
    attr_col = model_info['attribute_col']

    diagnostics = client.query(f"""
    SELECT * FROM `{dataset}.{model_name}_evaluation`
    """).to_dataframe()

    print("=== MODEL DIAGNOSTICS ===")
    print(diagnostics.to_string(index=False))

    if len(diagnostics) > 0 and 'aic' in diagnostics.columns:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].bar(diagnostics[attr_col], diagnostics['aic'], color='skyblue')
        axes[0].set_title('AIC (Lower is Better)')
        axes[0].set_ylabel('AIC')
        axes[0].tick_params(axis='x', rotation=45)

        axes[1].bar(diagnostics[attr_col], diagnostics['log_likelihood'], color='lightgreen')
        axes[1].set_title('Log Likelihood (Higher is Better)')
        axes[1].set_ylabel('Log Likelihood')
        axes[1].tick_params(axis='x', rotation=45)

        axes[2].bar(diagnostics[attr_col], diagnostics['variance'], color='salmon')
        axes[2].set_title('Model Variance')
        axes[2].set_ylabel('Variance')
        axes[2].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

    return diagnostics

def analyze_decomposition(model_info):
    """Analyze time series decomposition - adapted for short time series"""
    model_name = model_info['model_name']
    dataset = model_info['dataset']
    attr_col = model_info['attribute_col']

    decomp_query = f"""
    SELECT {attr_col}, time_series_timestamp, time_series_data,
           trend, seasonal_period_yearly, seasonal_period_weekly,
           holiday_effect, residual
    FROM `{dataset}.{model_name}_decomposition`
    ORDER BY {attr_col}, time_series_timestamp
    """
    decomposition = client.query(decomp_query).to_dataframe()
    decomposition['time_series_timestamp'] = pd.to_datetime(decomposition['time_series_timestamp'])

    groups = decomposition[attr_col].unique()

    for group in groups:
        group_data = decomposition[decomposition[attr_col] == group]

        # Determine which components have actual signal
        has_yearly = group_data['seasonal_period_yearly'].abs().max() > 0.1
        has_weekly = group_data['seasonal_period_weekly'].abs().max() > 0.1
        has_holiday = group_data['holiday_effect'].abs().max() > 0.1

        # Count meaningful components
        components = ['Original Data', 'Trend', 'Residual']
        if has_yearly: components.insert(2, 'Yearly Seasonal')
        if has_weekly: components.insert(2, 'Weekly Seasonal')
        if has_holiday: components.insert(-1, 'Holiday Effects')

        num_plots = len(components)
        fig, axes = plt.subplots(num_plots, 1, figsize=(15, 3 * num_plots))
        fig.suptitle(f'Time Series Decomposition for {group}', fontsize=16, fontweight='bold')

        plot_idx = 0

        # Original data
        axes[plot_idx].plot(group_data['time_series_timestamp'], group_data['time_series_data'],
                     color='blue', linewidth=1)
        axes[plot_idx].set_title('Original Data')
        axes[plot_idx].set_ylabel('Amount ($)')
        axes[plot_idx].grid(True, alpha=0.3)
        axes[plot_idx].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))
        plot_idx += 1

        # Trend
        axes[plot_idx].plot(group_data['time_series_timestamp'], group_data['trend'],
                     color='red', linewidth=2)
        axes[plot_idx].set_title('Trend Component')
        axes[plot_idx].set_ylabel('Trend ($)')
        axes[plot_idx].grid(True, alpha=0.3)
        axes[plot_idx].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))
        plot_idx += 1

        # Weekly seasonal (if meaningful)
        if has_weekly:
            axes[plot_idx].plot(group_data['time_series_timestamp'], group_data['seasonal_period_weekly'],
                         color='green', linewidth=1)
            axes[plot_idx].set_title('Weekly Seasonal Component')
            axes[plot_idx].set_ylabel('Seasonal ($)')
            axes[plot_idx].grid(True, alpha=0.3)
            plot_idx += 1

        # Yearly seasonal (if meaningful)
        if has_yearly:
            axes[plot_idx].plot(group_data['time_series_timestamp'], group_data['seasonal_period_yearly'],
                         color='orange', linewidth=1)
            axes[plot_idx].set_title('Yearly Seasonal Component')
            axes[plot_idx].set_ylabel('Seasonal ($)')
            axes[plot_idx].grid(True, alpha=0.3)
            plot_idx += 1

        # Holiday effects (if meaningful)
        if has_holiday:
            axes[plot_idx].plot(group_data['time_series_timestamp'], group_data['holiday_effect'],
                         color='purple', linewidth=1)
            axes[plot_idx].set_title('Holiday Effects')
            axes[plot_idx].set_ylabel('Holiday ($)')
            axes[plot_idx].grid(True, alpha=0.3)
            plot_idx += 1

        # Residual
        axes[plot_idx].plot(group_data['time_series_timestamp'], group_data['residual'],
                     color='gray', linewidth=1)
        axes[plot_idx].set_title('Residual (Unexplained Variation)')
        axes[plot_idx].set_ylabel('Residual ($)')
        axes[plot_idx].set_xlabel('Date')
        axes[plot_idx].grid(True, alpha=0.3)

        # Print which components were skipped
        skipped = []
        if not has_yearly: skipped.append('Yearly Seasonal')
        if not has_weekly: skipped.append('Weekly Seasonal')
        if not has_holiday: skipped.append('Holiday Effects')
        if skipped:
            print(f"\n  [{group}] Skipped flat components: {', '.join(skipped)}")
            print(f"  (Need more historical data to detect these patterns)")

        for ax in axes:
            ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

    return decomposition

def display_summary_statistics(model_info):
    """Display summary statistics"""
    model_name = model_info['model_name'].upper()
    dataset = model_info['dataset']
    attr_col = model_info['attribute_col'].upper()

    summary_query = f"""
    WITH SUMMARY_STATS AS (
      SELECT {attr_col}, COUNT(*) as HISTORICAL_DATA_POINTS,
        MIN(ds) as START_DATE, MAX(ds) as END_DATE,
        AVG(y) as AVG_HISTORICAL_AMOUNT, STDDEV(y) as STDDEV_HISTORICAL_AMOUNT
      FROM `{dataset}.{model_name}_FORECAST_DATA` GROUP BY {attr_col}
    ),
    FORECAST_SUMMARY AS (
      SELECT {attr_col}, AVG(FORECAST_VALUE) as AVG_FORECAST_AMOUNT,
        STDDEV(FORECAST_VALUE) as STDDEV_FORECAST_AMOUNT,
        MIN(FORECAST_TIMESTAMP) as FORECAST_START, MAX(FORECAST_TIMESTAMP) as FORECAST_END
      FROM `{dataset}.{model_name}_FORECAST_RESULTS` GROUP BY {attr_col}
    )
    SELECT hs.{attr_col}, hs.HISTORICAL_DATA_POINTS, hs.START_DATE, hs.END_DATE,
      hs.AVG_HISTORICAL_AMOUNT, hs.STDDEV_HISTORICAL_AMOUNT,
      fs.AVG_FORECAST_AMOUNT, fs.STDDEV_FORECAST_AMOUNT, fs.FORECAST_START, fs.FORECAST_END,
      CASE
        WHEN hs.AVG_HISTORICAL_AMOUNT = 0 THEN NULL
        ELSE (fs.AVG_FORECAST_AMOUNT - hs.AVG_HISTORICAL_AMOUNT) / hs.AVG_HISTORICAL_AMOUNT * 100
      END as PERCENT_CHANGE
    FROM SUMMARY_STATS hs
    LEFT JOIN FORECAST_SUMMARY fs ON hs.{attr_col} = fs.{attr_col}
    ORDER BY hs.{attr_col}
    """

    # Save to BigQuery table
    create_table_sql = f"""
    CREATE OR REPLACE TABLE `{dataset}.{model_name}_SUMMARY_STATS` AS
    {summary_query}
    """
    client.query(create_table_sql).result()
    print(f"Summary statistics saved to `{dataset}.{model_name}_summary_stats`")


    summary = client.query(summary_query).to_dataframe()

    print("=== SUMMARY STATISTICS ===")
    print(summary.to_string(index=False, float_format='{:,.2f}'.format))

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Forecast Summary Statistics', fontsize=16, fontweight='bold')

    x = np.arange(len(summary))
    width = 0.35

    axes[0, 0].bar(x - width/2, summary['avg_historical_amount'], width, label='Historical Avg', color='skyblue')
    axes[0, 0].bar(x + width/2, summary['avg_forecast_amount'], width, label='Forecast Avg', color='lightcoral')
    axes[0, 0].set_title('Historical vs Forecast Averages')
    axes[0, 0].set_ylabel('Amount ($)')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(summary[attr_col])
    axes[0, 0].legend()
    axes[0, 0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))

    bar_colors = ['green' if v >= 0 else 'red' for v in summary['percent_change']]
    axes[0, 1].bar(summary[attr_col], summary['percent_change'], color=bar_colors)
    axes[0, 1].set_title('Percent Change (Forecast vs Historical)')
    axes[0, 1].set_ylabel('Percent Change (%)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)

    axes[1, 0].bar(summary[attr_col], summary['historical_data_points'], color='gold')
    axes[1, 0].set_title('Historical Data Points')
    axes[1, 0].set_ylabel('Number of Data Points')
    axes[1, 0].tick_params(axis='x', rotation=45)

    axes[1, 1].bar(x - width/2, summary['stddev_historical_amount'], width, label='Historical StdDev', color='lightgreen')
    axes[1, 1].bar(x + width/2, summary['stddev_forecast_amount'], width, label='Forecast StdDev', color='orange')
    axes[1, 1].set_title('Standard Deviation Comparison')
    axes[1, 1].set_ylabel('Standard Deviation ($)')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(summary[attr_col])
    axes[1, 1].legend()
    axes[1, 1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))

    plt.tight_layout()
    plt.show()
    return summary


def run_complete_analysis(project_id, attribute_col, attribute_values, forecast_horizon=7, visuals = False):
    """Run the complete forecasting analysis end-to-end"""

    print("Starting BigQuery ML Forecast Analysis...")
    print("=" * 60)

    # Create model and tables
    model_info = create_forecast_model(project_id,attribute_col, attribute_values, forecast_horizon)

    # Summary Statistics
    print("\nGenerating Summary Statistics...")
    summary_stats = display_summary_statistics(model_info)

    if visuals:
      # Model Diagnostics
      print("\nAnalyzing Model Diagnostics...")
      diagnostics = analyze_model_diagnostics(model_info)

      # Time Series Decomposition
      print("\nAnalyzing Time Series Decomposition...")
      decomposition = analyze_decomposition(model_info)

      # Forecast Visualization
      print("\nCreating Forecast Visualization...")
      viz_data = plot_forecast_comparison(model_info)

      print("\nComplete Analysis Finished!")
      print("=" * 60)

      return {
          'model_info': model_info,
          'summary_stats': summary_stats,
          'diagnostics': diagnostics,
          'decomposition': decomposition,
          'visualization_data': viz_data
      }
    else:
      print("\nComplete Analysis Finished!")
      print("=" * 60)
      return {
          'model_info': model_info,
          'summary_stats': summary_stats
      }

## Data Exploration and Attribute Discovery

### Load Prepared Data

Load the transformed dataset for analysis:

In [ ]:
df_pos = bpd.read_gbq(f'{PROJECT_ID}.forecasting_us.POSITIVE_AMOUNTS')

In [ ]:
df_pos.columns

In [ ]:
df_pos["category"].value_counts()

In [ ]:
df_pos["commodity"] = df_pos["commodity"].replace("N/A", "Unknown Commodity")

In [ ]:
df_pos["commodity"].value_counts()

### Extract Unique Attribute Values

Discover the unique values for each organizational hierarchy level:

In [ ]:
group_list = df_pos['group_vp'].unique().to_list()
palmer_list = df_pos['palmer_vp'].unique().to_list()
division_list = df_pos['division'].unique().to_list()
region_list = df_pos['region'].unique().to_list()
account_list = df_pos['account'].unique().to_list()
commodity_list = df_pos['commodity'].unique().to_list()
category_list = df_pos['category'].unique().to_list()


## Analysis Execution

### Run Forecasting Analysis

Execute the complete forecasting analysis for each organizational level. Each analysis creates:

1. **Forecast Data**: Aggregated historical data by date and attribute
2. **ARIMA_PLUS Model**: Time series model with automatic parameter selection
3. **Forecast Results**: Future predictions with confidence intervals
4. **Model Evaluation**: Performance metrics and diagnostic information
5. **Visualization Data**: Combined historical and forecast data for plotting
6. **Decomposition**: Trend, seasonal, and holiday effect breakdown
7. **Summary Statistics**: Historical vs forecast comparison metrics

---

### Generated Outputs Summary

### Tables Created Per Analysis

Each forecasting analysis generates **6 comprehensive tables**:

1. **`{model_name}_FORECAST_DATA`** - Cleaned and aggregated historical data
2. **`{model_name}_FORECAST_RESULTS`** - Future predictions with confidence intervals
3. **`{model_name}_EVALUATION`** - Model performance metrics (AIC, log likelihood, etc.)
4. **`{model_name}_VISUALIZATION_DATA`** - Combined data for plotting
5. **`{model_name}_DECOMPOSITION`** - Time series component breakdown
6. **`{model_name}_SUMMARY_STATS`** - Historical vs forecast comparisons

### Key Metrics Provided

- **Forecast Accuracy**: AIC and log likelihood scores
- **Trend Analysis**: Long-term patterns and direction
- **Seasonal Patterns**: Yearly, weekly, and holiday effects
- **Confidence Intervals**: Upper and lower bounds for predictions
- **Percent Change**: Expected change vs historical averages
- **Data Quality**: Historical data points and variance measures

---

## Organizational Level Analysis

### Group Level Analysis

Forecast by **Group VP** - highest organizational level:

In [ ]:
results = run_complete_analysis(PROJECT_ID, 'group_vp', group_list, 1, 'group',visuals=False)

### Palmer Level Analysis

Forecast by **Palmer** - as in Palmer from Orale:

In [ ]:
results = run_complete_analysis(PROJECT_ID,'palmer_vp', palmer_list, 1, 'palmer',visuals=False)

### Division Level Analysis

Forecast by **Division** - operational divisions:

In [ ]:
results = run_complete_analysis(PROJECT_ID, 'division', division_list, 1, 'division',visuals=False)

### Region Level Analysis

Forecast by **Region** - geographical segmentation:

In [ ]:
results = run_complete_analysis(PROJECT_ID, 'region', region_list, 1, 'region',visuals=False)

### Account Level Analysis

Forecast by **Account** - detailed account-level analysis.

**Note**: This may generate many models and can be resource-intensive. Uncomment to run if needed.

In [ ]:
results = run_complete_analysis(PROJECT_ID, 'account', account_list, 1, 'account',visuals=False)

### Category Level Analysis

Forecast by **Account** - detailed account-level analysis.

**Note**: This may generate many models and can be resource-intensive. Uncomment to run if needed.

In [ ]:
results = run_complete_analysis(PROJECT_ID, 'category', category_list, 1, 'category',visuals=False)

### Modality Level Analysis

Forecast by **Account** - detailed account-level analysis.

**Note**: This may generate many models and can be resource-intensive. Uncomment to run if needed.

### Vendor Level Analysis

Forecast by **Account** - detailed account-level analysis.

**Note**: This may generate many models and can be resource-intensive. Uncomment to run if needed.

### Commodity Level Analysis

Forecast by **Account** - detailed account-level analysis.

**Note**: This may generate many models and can be resource-intensive. Uncomment to run if needed.

In [ ]:
results = run_complete_analysis(PROJECT_ID, 'commodity', commodity_list, 1, 'commodity',visuals=False)

## Cleanup

### Close BigQuery Session

Clean up resources and close the BigQuery session:

In [ ]:
bpd.close_session()